# L14 demo: Bayesian optimization and active learning

Each expensive evaluation is a simulation run or a physical experiment, and the budget is
a few dozen. The job is to choose the next one well. We build the Bayesian-optimization
loop twice: first by hand so the acquisition math is visible, then in BoTorch on an
engineering objective. Then we do the honest things: beat a random-search baseline over
many seeds, and run one active-learning step.

> Companion notes: [`notes.md`](notes.md). Dataset: NASA airfoil self-noise, carried over
> from L13.

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # GP fits emit convergence chatter

import io, urllib.request, zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.ensemble import GradientBoostingRegressor

SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 12})
CMU_RED, BLUE, GREEN, BAND = '#c41230', '#1f5c99', '#2b7a4b', '#c9dbec'

## 1. The loop by hand on a 1-D function

The [Forrester function](https://www.sciencedirect.com/book/9780470060681) is a standard
one-dimensional test case with a global minimum near $x = 0.757$ and a shallow local
minimum on the left to trap a greedy optimizer. We minimize it.

In [ ]:
def forrester(x):
    x = np.asarray(x, dtype=float)
    return (6 * x - 2) ** 2 * np.sin(12 * x - 4)

grid = np.linspace(0, 1, 400)
x_star = grid[forrester(grid).argmin()]
print(f'true minimum near x = {x_star:.3f}, f = {forrester(x_star):.3f}')

### The acquisition, written out

Two functions, both for **minimization**. Expected improvement scores how much we expect
to beat the incumbent `best`; the lower confidence bound is `mu - kappa * sigma`, with
`kappa` the explore/exploit dial. Watch the sign: the improvement is `best - mu`, because
lower is better here.

In [ ]:
def expected_improvement(mu, sigma, best, xi=0.01):
    sigma = np.maximum(sigma, 1e-9)
    imp = best - mu - xi            # improvement BELOW the incumbent (minimization)
    z = imp / sigma
    return imp * norm.cdf(z) + sigma * norm.pdf(z)

def lower_confidence_bound(mu, sigma, kappa=2.0):
    return mu - kappa * sigma        # we will MINIMIZE this

def fit_gp(x, y):
    kernel = (ConstantKernel(1.0, (1e-2, 1e2))
              * Matern(0.15, (0.05, 0.5), nu=2.5)
              + WhiteKernel(1e-4, (1e-6, 1e-1)))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                  n_restarts_optimizer=4, random_state=SEED)
    return gp.fit(x.reshape(-1, 1), y)

A quick sign check before we trust it: expected improvement should be higher at a point
we believe is good than at one we believe is bad.

In [ ]:
x0 = np.array([0.0, 0.33, 0.66, 1.0])          # a space-filling start
y0 = forrester(x0)
gp0 = fit_gp(x0, y0)
mu0, sd0 = gp0.predict(grid.reshape(-1, 1), return_std=True)
ei0 = expected_improvement(mu0, sd0, y0.min())
good, bad = grid[mu0.argmin()], grid[mu0.argmax()]
print(f'EI at the predicted-good x={good:.2f}: {expected_improvement(*gp0.predict([[good]], return_std=True), y0.min())[0]:.4f}')
print(f'EI at the predicted-bad  x={bad:.2f}: {expected_improvement(*gp0.predict([[bad]], return_std=True), y0.min())[0]:.4f}')
assert ei0.max() > 0, 'acquisition is flat: something is wrong'
print('sign check ok: EI favours the promising region')

### Run the loop

Fit the GP, maximize EI, evaluate the true function there, repeat. From a start that
misses the global basin, EI walks into it in a handful of steps.

In [ ]:
x, y = x0.copy(), y0.copy()
picks = []
for it in range(6):
    gp = fit_gp(x, y)
    mu, sd = gp.predict(grid.reshape(-1, 1), return_std=True)
    x_next = grid[expected_improvement(mu, sd, y.min()).argmax()]
    picks.append(x_next)
    x = np.append(x, x_next)
    y = np.append(y, forrester(x_next))

print('proposed points:', [round(float(p), 3) for p in picks])
print(f'best found: x = {x[y.argmin()]:.3f}, f = {y.min():.3f}  (true {forrester(x_star):.3f})')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
gp = fit_gp(x, y)
mu, sd = gp.predict(grid.reshape(-1, 1), return_std=True)
ax.plot(grid, forrester(grid), color='0.8', lw=2, label='true objective')
ax.fill_between(grid, mu - 1.96 * sd, mu + 1.96 * sd, color=BAND, alpha=0.8)
ax.plot(grid, mu, color=BLUE, lw=2, label='GP mean')
ax.scatter(x0, y0, color='0.4', s=45, zorder=4, label='initial design')
ax.scatter(x[len(x0):], y[len(x0):], color=CMU_RED, s=45, zorder=5, label='BO evaluations')
ax.axvline(x_star, color='k', ls=':', lw=1.2, label='true optimum')
ax.set_xlabel('design variable x'); ax.set_ylabel('f(x)'); ax.legend(fontsize=9)
ax.set_title('Bayesian optimization on the Forrester function'); plt.show()

## 2. The same loop in BoTorch on an engineering objective

Now the airfoil self-noise data from L13. We fit a gradient-boosted emulator to all 1503
rows and treat *that* as the expensive experiment: the optimizer never sees it, it only
gets to query it. The design space is the box of observed operating conditions, and we
want the quietest one (lowest sound pressure level).

In [ ]:
CACHE = Path('.cache'); CACHE.mkdir(exist_ok=True)
URL = 'https://archive.ics.uci.edu/static/public/291/airfoil+self+noise.zip'
local = CACHE / 'airfoil_self_noise.dat'
if not local.exists():
    with urllib.request.urlopen(URL) as r:
        local.write_bytes(zipfile.ZipFile(io.BytesIO(r.read())).read('airfoil_self_noise.dat'))
data = np.loadtxt(local)
X_all, y_all = data[:, :5], data[:, 5]
oracle = GradientBoostingRegressor(random_state=SEED, n_estimators=400,
                                   max_depth=3, learning_rate=0.05).fit(X_all, y_all)
lo, hi = X_all.min(0), X_all.max(0)
def experiment(pt):
    return float(oracle.predict(np.asarray(pt).reshape(1, -1))[0])

floor = oracle.predict(np.random.default_rng(1).uniform(lo, hi, (200_000, 5))).min()
print(f'best SPL the emulator allows over the box: {floor:.2f} dB')

BoTorch maximizes by convention, so to minimize noise we optimize on the negative SPL.
The `Normalize` and `Standardize` transforms handle input and output scaling; the loop is
the same four steps as above with `SingleTaskGP` and `qExpectedImprovement` in place of
the hand-written pieces.

In [ ]:
import torch
from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.acquisition import qExpectedImprovement
from botorch.optim import optimize_acqf

torch.manual_seed(SEED)
td = {'dtype': torch.double}
bounds = torch.tensor(np.vstack([lo, hi]), **td)

init = np.random.default_rng(SEED).uniform(lo, hi, (6, 5))
X = torch.tensor(init, **td)
Y = torch.tensor([[-experiment(p)] for p in init], **td)   # negate: maximize -SPL

for it in range(14):
    gp = SingleTaskGP(X, Y, input_transform=Normalize(d=5, bounds=bounds),
                      outcome_transform=Standardize(m=1))
    fit_gpytorch_mll(ExactMarginalLogLikelihood(gp.likelihood, gp))
    acqf = qExpectedImprovement(gp, best_f=Y.max())
    cand, _ = optimize_acqf(acqf, bounds=bounds, q=1, num_restarts=5, raw_samples=64)
    X = torch.cat([X, cand])
    Y = torch.cat([Y, torch.tensor([[-experiment(cand.numpy().ravel())]], **td)])

best_spl = -float(Y.max())
print(f'BoTorch best SPL after 20 evaluations: {best_spl:.2f} dB  (floor {floor:.2f} dB)')

## 3. Honest evaluation: Bayesian optimization vs random search

Bayesian optimization is stochastic, so a single run proves nothing. We run both
strategies over many seeds, on the same emulator and the same budget, and plot the median
best-so-far with an interquartile band. We use the transparent scikit-learn optimizer here
because it is fast enough to repeat 20 times.

In [ ]:
def bo_sklearn(seed, budget=22, n_init=4):
    r = np.random.default_rng(1000 + seed)
    Xd = r.uniform(lo, hi, (n_init, 5)); yd = np.array([experiment(p) for p in Xd])
    best = [yd.min()]
    ker = (ConstantKernel(1.0, (1e-2, 1e3)) * Matern(np.ones(5), nu=2.5)
           + WhiteKernel(1.0, (1e-3, 1e3)))
    for _ in range(budget - n_init):
        g = GaussianProcessRegressor(ker, normalize_y=True, random_state=seed)
        m, s = Xd.mean(0), Xd.std(0) + 1e-9
        g.fit((Xd - m) / s, yd)
        pool = r.uniform(lo, hi, (1500, 5))
        mu, sd = g.predict((pool - m) / s, return_std=True)
        nxt = pool[expected_improvement(mu, sd, yd.min()).argmax()]
        Xd = np.vstack([Xd, nxt]); yd = np.append(yd, experiment(nxt)); best.append(yd.min())
    return best

def random_search(seed, budget=22, n_init=4):
    r = np.random.default_rng(1000 + seed)
    yd = np.array([experiment(p) for p in r.uniform(lo, hi, (n_init, 5))]); best = [yd.min()]
    for _ in range(budget - n_init):
        yd = np.append(yd, experiment(r.uniform(lo, hi, 5))); best.append(yd.min())
    return best

N = 20
bo = np.array([bo_sklearn(s) for s in range(N)])
rs = np.array([random_search(s) for s in range(N)])
xs = np.arange(4, 4 + bo.shape[1])
print(f'after 22 evals  BO median {np.median(bo[:, -1]):.2f} dB   random {np.median(rs[:, -1]):.2f} dB')

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for curves, color, name in [(bo, CMU_RED, 'Bayesian optimization'), (rs, '0.5', 'random search')]:
    ax.plot(xs, np.median(curves, 0), color=color, lw=2.4, label=name)
    q1, q3 = np.percentile(curves, [25, 75], axis=0)
    ax.fill_between(xs, q1, q3, color=color, alpha=0.15)
ax.axhline(floor, color='k', ls=':', lw=1.3)
ax.set_xlabel('expensive evaluations spent'); ax.set_ylabel('lowest SPL found (dB)')
ax.set_title(f'median of {N} seeds, with interquartile band'); ax.legend(); plt.show()

The bands are the message. A single seed of either method could land anywhere inside
them, so any claim that Bayesian optimization 'worked' has to be a claim about the
distribution, not about one lucky run.

## 4. One active-learning step

Same machinery, different goal: improve the surrogate everywhere rather than find one
optimum. We query the point of highest posterior uncertainty. Holding the hyperparameters
fixed, conditioning on that one observation can only shrink the posterior variance.

In [ ]:
xa = np.array([0.08, 0.2, 0.32, 0.44, 0.9]); ya = forrester(xa)
g = fit_gp(xa, ya)
mu, sd = g.predict(grid.reshape(-1, 1), return_std=True)
x_query = grid[sd.argmax()]
before = np.trapezoid(sd, grid)

xb = np.append(xa, x_query); yb = np.append(ya, forrester(x_query))
g2 = GaussianProcessRegressor(kernel=g.kernel_, optimizer=None, normalize_y=True).fit(xb.reshape(-1, 1), yb)
mu2, sd2 = g2.predict(grid.reshape(-1, 1), return_std=True)
after = np.trapezoid(sd2, grid)
print(f'query at x = {x_query:.3f}; total posterior SD {before:.3f} -> {after:.3f} ({100*(before-after)/before:.1f}% lower)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (m, s, xx, yy, ttl, q) in zip(axes, [
        (mu, sd, xa, ya, 'before: widest gap between 0.44 and 0.9', x_query),
        (mu2, sd2, xb, yb, 'after one query at the most uncertain point', None)]):
    ax.plot(grid, forrester(grid), color='0.8', lw=2)
    ax.fill_between(grid, m - 1.96 * s, m + 1.96 * s, color=BAND, alpha=0.85)
    ax.plot(grid, m, color=BLUE, lw=2); ax.scatter(xx, yy, color='0.3', s=40, zorder=4)
    if q is not None: ax.axvline(q, color=CMU_RED, ls='--', lw=1.6)
    ax.set_title(ttl, fontsize=11); ax.set_xlabel('design variable x')
axes[0].set_ylabel('f(x)'); plt.show()

---

## Takeaway

We built the Bayesian-optimization loop by hand to see the acquisition math, then in
BoTorch to see the production tool, and both are the same four steps: fit a probabilistic
surrogate, maximize an acquisition, evaluate, update. The honest-evaluation section is the
part to internalize: report over many seeds against a random-search baseline, because a
single run is a sample of one. Active learning is the same loop aimed at the model instead
of the optimum, spending each query where the surrogate is most uncertain. This is the
design-loop payoff the miniproject (A7) asks you to demonstrate.